# PulseOps: Closed-Loop AI Incident Reliability & Recovery
### 24-Hour IBM Z Datathon Prototype

**Core Pitch:** *PulseOps doesn't just detect incidents. It detects, responds, and proves whether recovery worked.*

---
### Closed-Loop Architecture
1. **Live Telemetry:** In-memory payment service simulator generating real-time metrics (`arrival_rate`, `service_time_ms`, `queue_depth`, `latency_ms`, `error_rate`, `cpu_pct`).
2. **Streaming Robust Detector:** Rolling Median/MAD/IQR baseline with scale floors, spike & sustained shift detection, persistence, and cooldown.
3. **Evidence-Grounded Diagnosis:** Distinguishes observed facts from hypotheses; recommends bounded actions.
4. **Safe Remediation Engine:** Allowlisted actions (`scale_workers`, `throttle_intake`, `snapshot_state`), strict bounds, cooldowns, human approval, and dry-run mode.
5. **Recovery Verification:** Measures post-action telemetry against baseline recovery criteria. Proves recovery or reports honest failure.

> **Zero External Dependencies:** Built 100% on Python Standard Library. Zero binary wheels, zero pip installs, fully native on LinuxONE (`s390x`).

## 1. Environment & Architecture Check
Verify that we are running on standard library without external dependencies, and inspect system architecture.

In [ ]:
import sys, os, platform

# Ensure pulseops package is in path
sys.path.insert(0, os.path.abspath("."))

print("[Environment Check]")
print(f"- Python Version : {sys.version.split()[0]}")
print(f"- System OS      : {platform.system()} ({platform.release()})")
print(f"- Architecture   : {platform.machine()} (IBM Z LinuxONE is typically s390x)")
print(f"- Working Dir    : {os.getcwd()}")
print("- Zero-dependency: Ready to run with standard library only!")

## 2. Initialize Telemetry Simulator & Engine
We initialize the `PulseOpsEngine` which performs warm-up (90 simulated seconds) to establish robust statistical baselines.

In [ ]:
from pulseops.engine import PulseOpsEngine
from pulseops.remediation import RemediationConfig

# Initialize engine with standard configuration
engine = PulseOpsEngine()

state = engine.state()
print("Status after warm-up:")
print(f"- Health        : {state['health'].upper()}")
print(f"- Workers       : {state['workers']}")
print(f"- Active Alerts : {len(state['alerts'])}")
print("- Telemetry Baselines learned:")
for m, b in state['baselines'].items():
    print(f"    {m:15s} -> median={b['baseline']:.2f}, scale={b['scale']:.2f}")

## 3. Inject Controlled Workload Spike
We inject a 2.0x load spike into the simulated payment service. Notice how arrival rate spikes and queues begin to build.

In [ ]:
print("Injecting fault: load_spike (magnitude: 2.0x)...")
engine.inject_fault("load_spike", 2.0)

# Tick simulation until detector identifies anomaly
incident = None
for step in range(30):
    engine.tick()
    incident = engine.active_incident()
    if incident:
        print(f"-> Incident opened at tick {step+1} (t={engine.now:.1f}s)!")
        break

print(f"Incident ID: {incident['id']}")
print(f"Incident State: {incident['state']}")

## 4. Evidence-Grounded Diagnosis
PulseOps diagnoses the root cause using correlated metrics. It cleanly distinguishes **Observed Facts** from **Hypotheses**.

In [ ]:
diag = incident["diagnosis"]
print(f"SUMMARY: {diag['summary']}\n")
print("OBSERVED FACTS:")
for f in diag["facts"]:
    print(f"  [x] {f}")
print("\nHYPOTHESES:")
for h in diag["hypotheses"]:
    print(f"  [?] {h['label']} (confidence: {h['confidence']})")
    for s in h.get('supporting', []):
        print(f"      + Supporting: {s}")
    for ne in h.get('not_established', []):
        print(f"      - Not established from telemetry: {ne}")
print("\nRECOMMENDED REMEDIATION:")
print(f"  Action : {diag['recommendation']['action']}")
print(f"  Params : {diag['recommendation']['params']}")
print(f"  Reason : {diag['recommendation']['rationale']}")

## 5. Human-in-the-Loop Operator Approval
Autonomous action without bounds or approval can cause catastrophic cascade failures. PulseOps stages the recommended action and requires human sign-off.

In [ ]:
pending_id = incident["pending_action"]
print(f"Pending action awaiting approval: {pending_id}")

# Operator approves the proposed remediation
res = engine.approve(pending_id, who="lead-sre-judge")
print(f"Execution result: {res}")

# Advance simulation during recovery verification window
for _ in range(60):
    engine.tick()
    if engine.active_incident() is None:
        break

inc_final = engine.incidents[incident["id"]]
print(f"Final Incident State: {inc_final['state'].upper()}")

## 6. Mathematical Recovery Verification
**Crucial Datathon Requirement:** PulseOps *never* assumes recovery just because an action was executed. It collects post-action telemetry and compares against statistical recovery criteria.

In [ ]:
v = inc_final["verification"]
print(f"VERIFICATION VERDICT: {v['verdict'].upper()}")
print(f"Reason: {v['reason']}\n")
print(f"{'Metric':15s} | {'Before (Peak)':14s} | {'After (Mean)':14s} | {'Threshold':12s} | {'Pass?':5s}")
print("-" * 70)
for metric, mdata in v["metrics"].items():
    bp = f"{mdata['before_peak']:.2f}" if mdata['before_peak'] is not None else "n/a"
    am = f"{mdata['after_mean']:.2f}" if mdata['after_mean'] is not None else "n/a"
    th = f"{mdata['threshold']:.2f}" if mdata['threshold'] is not None else "n/a"
    print(f"{metric:15s} | {bp:14s} | {am:14s} | {th:12s} | {str(mdata['passed']):5s}")

## 7. Incident Timeline Audit Log
All actions, automated snapshots, diagnostics, approvals, and outcomes are permanently logged in an audit trail.

In [ ]:
print("Complete Incident Timeline:")
for ev in inc_final["timeline"]:
    print(f"  [t={ev['t']:5.1f}s] {ev['event'].upper():12s} : {ev['detail']}")

## 8. Honest Failure Handling: Degradation that Cannot Fully Recover
What happens if downstream dependencies slow down 3x? Scaling workers cannot reduce downstream latency below physical limits.
PulseOps honestly reports **`partially_recovered`** rather than claiming false success!

In [ ]:
engine_fail = PulseOpsEngine()
print("Injecting severe downstream latency degradation (3.0x)...")
engine_fail.inject_fault("slow_downstream", 3.0)
for _ in range(30):
    engine_fail.tick()
    if engine_fail.active_incident():
        break

inc2 = engine_fail.active_incident()
print(f"Incident {inc2['id']} opened. Diagnosis: {inc2['diagnosis']['summary']}")
print(f"Recommended: {inc2['diagnosis']['recommendation']['action']}")
engine_fail.approve(inc2["pending_action"], who="operator")

# Tick past the verification window (90s timeout)
for _ in range(120):
    engine_fail.tick()
    if engine_fail.active_incident() is None:
        break

inc2_res = engine_fail.incidents[inc2['id']]
print(f"\nVerdict: {inc2_res['verification']['verdict'].upper()}")
print(f"Explanation: {inc2_res['verification']['reason']}")
print(f"Notes: {inc2_res['notes']}")

## 9. Launching the Interactive Web Dashboard
You can run the web dashboard directly from Jupyter or the command line (`python -m pulseops.server --port 8000`).
The dashboard is 100% self-contained HTML/JS/Canvas with no external CDN requirements.

In [ ]:
print("To run the live interactive dashboard web UI:")
print("  python -m pulseops.server --port 8000")
print("Then open: http://127.0.0.1:8000 (or use SSH tunnel if remote)")